# Droideka

Bot Droideka

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

import sklearn as sk
from sklearn import model_selection
from sklearn import ensemble
from sklearn import metrics


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 0. Lectura de datos

In [ ]:
DIR = "/content/drive/MyDrive/datos_maestria_2026"
engine = sqlite3.connect(f"{DIR}/entrenamiento.db")
df_ent = pd.read_sql("SELECT * FROM entrenamiento", engine, index_col='id')
df_ap = pd.read_csv(f"{DIR}/a_predecir.csv", index_col="id")

In [ ]:
# cantidad de filas y columnas
df_ent.shape

In [ ]:
df_ent.columns

## 1. Entender los datos (AID)

## 2. Limpiar y transformar los datos (DM)

In [ ]:
df_ent["property_type"] = df_ent["property_type"].replace('departamentos','departamento')
df_ent["property_type"] = df_ent["property_type"].replace('casas','casa')


In [ ]:
df_ent.property_type.value_counts()

## 2.1. Filtrado de datos



In [ ]:
# selección de datos
tiene_precio = df_ent["price"].notna()
moneda_dolares = df_ent["currency_type"] == 'dolares'
localidades = df_ent["location_1"].isin(["Capital Federal", "Ciudad Autónoma de Buenos Aires", 'CABA'])
tipo_operacion = df_ent["operation_type"] == 'venta'
tipo_propiedad = df_ent["property_type"].isin(["departamento", "casa"])  # ya normalizados en celda anterior

df_ent = df_ent.loc[tiene_precio & moneda_dolares & localidades & tipo_operacion & tipo_propiedad]
df_ent.shape

In [ ]:
# Quitamos filas con "ambiente" en location_2
df_ent = df_ent[~df_ent["location_2"].str.contains("ambiente", case=False, na=False)]

In [ ]:
# Eliminar propiedades con coordenadas fuera de CABA
coords_otra_ciudad = (
    (df_ent["lat"] != 0) & (df_ent["lon"] != 0) &
    ~(df_ent["lat"].between(-34.73, -34.52) & df_ent["lon"].between(-58.54, -58.33))
)
df_ent = df_ent[~coords_otra_ciudad]

# Imputar lat/lon=0 con centroide por sub-barrio → barrio → CABA general
coords_ok = df_ent["lat"].between(-34.73, -34.52) & df_ent["lon"].between(-58.54, -58.33)

centroide_l3 = df_ent[coords_ok].groupby("location_3")[["lat", "lon"]].mean()
centroide_l2 = df_ent[coords_ok].groupby("location_2")[["lat", "lon"]].mean()
caba_lat = df_ent.loc[coords_ok, "lat"].mean()
caba_lon = df_ent.loc[coords_ok, "lon"].mean()
mask = ~coords_ok

# Imputar lat/lon con centroide por sub-barrio → barrio → CABA general
df_ent.loc[mask, "lat"] = (
    df_ent.loc[mask, "location_3"].map(centroide_l3["lat"])
    .fillna(df_ent.loc[mask, "location_2"].map(centroide_l2["lat"]))
    .fillna(caba_lat)
)
df_ent.loc[mask, "lon"] = (
    df_ent.loc[mask, "location_3"].map(centroide_l3["lon"])
    .fillna(df_ent.loc[mask, "location_2"].map(centroide_l2["lon"]))
    .fillna(caba_lon)
)

print(f"Filas tras limpieza de coords: {df_ent.shape[0]}")

## 2.2. Tratamiento de valores atípicos

In [ ]:
Q1 = df_ent["price"].quantile(0.25)
Q3 = df_ent["price"].quantile(0.75)
IQR = Q3 - Q1

df_ent = df_ent[
    (df_ent["price"] >= Q1 - 1.5 * IQR) &
    (df_ent["price"] <= Q3 + 1.5 * IQR)
]

## 2.3. Imputación de valores perdidos

In [ ]:
# Guardar publication_date antes del fillna (el regex de año/mes necesita el string original)
_pub = df_ent["publication_date"].copy()

# TODO: Mejorar la estrategia de imputación.
df_ent = df_ent.fillna(0)

## 2.4. Creación de nuevos atributos

In [ ]:
import re

def parse_ar(s):
    """Parsea números en notación argentina: punto=miles, coma=decimal."""
    if pd.isna(s): return float('nan')
    s = str(s).strip().replace('\xa0', '').replace(' ', '')
    if ',' in s and '.' in s:
        s = s.replace('.', '').replace(',', '.')   # 1.081,87 → 1081.87
    elif ',' in s:
        s = s.replace(',', '.')                    # 38,92 → 38.92
    elif re.match(r'^\d+\.\d{3}$', s):
        s = s.replace('.', '')                     # 1.200 → 1200
    try: return float(s)
    except: return float('nan')

def extraer(patron):
    return texto.str.extract(patron, expand=False).apply(parse_ar)

# creamos otras variables a partir de texto libre (features y description)
texto = (
    df_ent["features"].fillna("").astype(str) + " " +
    df_ent["description"].fillna("").astype(str)
)

df_ent["bedrooms"]           = extraer(r"([\d]+)\s*(?:dormitorios?|habitaciones?)")
df_ent["bathrooms"]          = extraer(r"([\d]+)\s*(?:baños?|banos?)")
df_ent["rooms"]              = extraer(r"([\d]+)\s*(?:ambientes?|rooms?)")
df_ent["surface_total_m2"]   = extraer(r"([\d]+(?:[.,]\d+)?)\s*m²?")
df_ent["surface_covered_m2"] = extraer(r"([\d]+(?:[.,]\d+)?)\s*m²?\s*(?:cubiertos?|cubierta)")
df_ent["tiene_garage"]       = texto.str.contains(r"garage|cochera|parking", case=False, na=False).astype(int)
df_ent["piso"]               = extraer(r"[Pp]iso\s*([\d]+)")
df_ent["antiguedad"]         = extraer(r"antiguedad[:\s]*([\d]+)")
df_ent["es_estreno"]         = texto.str.contains(r"a estrenar|estreno|0 km", case=False, na=False).astype(int)

# monoambiente → rooms = 1
es_mono = texto.str.contains(r"monoambiente", case=False, na=False)
df_ent.loc[es_mono & df_ent["rooms"].isna(), "rooms"] = 1

# planta baja → piso = 0
es_pb = texto.str.contains(r"planta baja", case=False, na=False)
df_ent.loc[es_pb & df_ent["piso"].isna(), "piso"] = 0

In [ ]:
# Si es casa o departamento y no tiene baños, asumimos que tiene 1 baño (muchas veces no se especifica)
mask = (
    df_ent["property_type"].str.lower().isin(["casa", "departamento"]) &
    (df_ent["bathrooms"].isna() | (df_ent["bathrooms"] == 0))
)
df_ent.loc[mask, "bathrooms"] = 1

In [ ]:
# dummies de tipo de propiedad en ent y ap
df_ent["dummy_property_type"] = df_ent["property_type"]
df_ap["dummy_property_type"] = df_ap["property_type"]
df_ent = pd.get_dummies(df_ent, columns=["dummy_property_type"], prefix="is")
df_ap = pd.get_dummies(df_ap, columns=["dummy_property_type"], prefix="is", )

In [ ]:
# arreglar las varibles creadas
df_ent.loc[:, "bedrooms"] = df_ent["bedrooms"].fillna(df_ent["rooms"] - 1)
df_ent.loc[:, "rooms"] = df_ent["rooms"].fillna(df_ent["bedrooms"] + 1)
df_ent.loc[:, "bathrooms"] = df_ent["bathrooms"].fillna(df_ent["rooms"] - df_ent["bedrooms"])

df_ent["surface_total_m2"] = df_ent["surface_total_m2"].fillna(df_ent["surface_covered_m2"])
df_ent["surface_covered_m2"] = df_ent["surface_covered_m2"].fillna(df_ent["surface_total_m2"])

# crear variable de relacion superficie cubierta/superficie total
df_ent["pct_surface_covered"] = df_ent["surface_covered_m2"] / df_ent["surface_total_m2"]

# para nans
df_ent["rooms"] = df_ent["rooms"].fillna(1)
df_ent["bedrooms"] = df_ent["bedrooms"].fillna(1)

# superficie por ambiente
df_ent["m2_por_ambiente"] = df_ent["surface_total_m2"] / df_ent["rooms"].replace(0, float("nan"))

# distancia al Obelisco (proxy de centralidad)
OBELISCO_LAT, OBELISCO_LON = -34.6037, -58.3816
df_ent["dist_obelisco"] = np.sqrt(
    (df_ent["lat"] - OBELISCO_LAT) ** 2 + (df_ent["lon"] - OBELISCO_LON) ** 2
)

# año y mes de publicación
months_map = {"ene":1,"feb":2,"mar":3,"abr":4,"may":5,"jun":6,"jul":7,"ago":8,"sep":9,"oct":10,"nov":11,"dic":12}
df_ent["pub_year"]  = pd.to_numeric(_pub.str.extract(r"(\d{4})")[0], errors="coerce").fillna(0).astype(int)
df_ent["pub_month"] = _pub.str.lower().str.extract(r"(ene|feb|mar|abr|may|jun|jul|ago|sep|oct|nov|dic)")[0].map(months_map).fillna(0).astype(int)


In [ ]:
# Outliers de superficie con límites arbitrarios
# < 8 m²: error de parseo o garaje suelto. > 1500 m²: imposible en CABA residencial.
antes = df_ent.shape[0]
mask_sup_ok = df_ent["surface_total_m2"].isna() | df_ent["surface_total_m2"].between(8, 1500)
df_ent = df_ent[mask_sup_ok]
print(f"surface_total_m2: límites [8, 1500] m² — eliminadas {antes - df_ent.shape[0]} filas")


In [ ]:
# creación de precio por m²
df_ent["price_m2"] = df_ent["price"] / df_ent["surface_total_m2"].replace(0, float("nan"))

In [ ]:
# Guardamos location_2, location_3 y property_type antes del select_dtypes (son strings)

_location_2    = df_ent["location_2"].copy()
_location_3    = df_ent["location_3"].copy()
_property_type = df_ent["property_type"].copy()

# target encoding: precio_m2 promedio por tipo de propiedad y barrio (location_2)
gb = (
    df_ent.assign(_pt=_property_type, _l2=_location_2)
    .groupby(["_pt", "_l2"])["price_m2"]
    .mean()
    .rename("price_m2_mean")
)
df_ent["price_m2_mean"] = list(zip(_property_type, _location_2))
df_ent["price_m2_mean"] = df_ent["price_m2_mean"].map(gb).fillna(gb.mean())
df_ent["valor_estimado"] = df_ent["price_m2_mean"] * df_ent["surface_total_m2"]

# target encoding: precio_m2 promedio por tipo de propiedad y sub-barrio (location_3)
gb_l3 = (
    df_ent.assign(_pt=_property_type, _l3=_location_3)
    .groupby(["_pt", "_l3"])["price_m2"]
    .mean()
    .rename("price_m2_mean_l3")
)
df_ent["price_m2_mean_l3"] = list(zip(_property_type, _location_3))
df_ent["price_m2_mean_l3"] = df_ent["price_m2_mean_l3"].map(gb_l3).fillna(gb_l3.mean())
df_ent["valor_estimado_l3"] = df_ent["price_m2_mean_l3"] * df_ent["surface_total_m2"]

# Outliers de price_m2 con límites duros (aplicado después del encoding para no sesgar los promedios)
antes = df_ent.shape[0]
filtros = [
    (df_ent["property_type"] == "departamento") & (df_ent["location_3"] == "Puerto Madero") & df_ent["price_m2"].between(5000, 7000),
    (df_ent["property_type"] == "departamento") & (df_ent["location_3"] != "Puerto Madero") & df_ent["price_m2"].between(2000, 3000),
    (df_ent["property_type"] == "casa") & (df_ent["price_m2"].between(1500, 3000)),
]

# aplicar sucesivamente cada filtro creando un dataframe que concatene todos los registros filtrados
dfs_filtrados = []
for filtro in filtros:
    dfs_filtrados.append(df_ent[filtro])
df_ent = pd.concat(dfs_filtrados)

print(f"price_m2: límites [1500, 9000] $/m² — eliminadas {antes - df_ent.shape[0]} filas")
print(f"Filas totales: {df_ent.shape[0]}")

## 3. Selección de datos para modelado (DM)

In [ ]:
# La creación de modelos requiere que todo el dataframe sea numérico
df_ent = df_ent.select_dtypes(include=['number', 'bool'])

cols_excluir = ["price_m2", "location_4"]

X = df_ent[df_ent.columns.drop(["price"] + [c for c in cols_excluir if c in df_ent.columns])]
y = df_ent["price"]

## 4. Entrenamiento de modelos (AA)

In [ ]:
# Entrenamos con todos los datos usando los mejores hiperparámetros
hp_opts_v2 = {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 1000}
reg = sk.ensemble.RandomForestRegressor(**hp_opts_v2, n_jobs=-1, random_state=42)
reg.fit(X, y)
print("Modelo entrenado con", X.shape[0], "filas y", X.shape[1], "features")

## 5. Replicación de tratamiento de datos en el dataset a predecir

In [ ]:
ap_index = df_ap.index.copy()

# Normalizar property_type
df_ap["property_type"] = df_ap["property_type"].replace({"departamentos": "departamento", "casas": "casa"})

# Imputar coordenadas usando centroides calculados sobre df_ent
mask_ap = ~(df_ap["lat"].between(-34.73, -34.52) & df_ap["lon"].between(-58.54, -58.33))
df_ap.loc[mask_ap, "lat"] = (
    df_ap.loc[mask_ap, "location_3"].map(centroide_l3["lat"])
    .fillna(df_ap.loc[mask_ap, "location_2"].map(centroide_l2["lat"]))
    .fillna(caba_lat)
)
df_ap.loc[mask_ap, "lon"] = (
    df_ap.loc[mask_ap, "location_3"].map(centroide_l3["lon"])
    .fillna(df_ap.loc[mask_ap, "location_2"].map(centroide_l2["lon"]))
    .fillna(caba_lon)
)

# Extraer features del texto usando parse_ar (maneja notación argentina)
texto_ap = df_ap["features"].fillna("").astype(str) + " " + df_ap["description"].fillna("").astype(str)
def extraer_ap(patron):
    return texto_ap.str.extract(patron, expand=False).apply(parse_ar)

df_ap["bedrooms"]           = extraer_ap(r"([\d]+)\s*(?:dormitorios?|habitaciones?)")
df_ap["bathrooms"]          = extraer_ap(r"([\d]+)\s*(?:baños?|banos?)")
df_ap["rooms"]              = extraer_ap(r"([\d]+)\s*(?:ambientes?|rooms?)")
df_ap["surface_total_m2"]   = extraer_ap(r"([\d]+(?:[.,]\d+)?)\s*m²?")
df_ap["surface_covered_m2"] = extraer_ap(r"([\d]+(?:[.,]\d+)?)\s*m²?\s*(?:cubiertos?|cubierta)")
df_ap["tiene_garage"]       = texto_ap.str.contains(r"garage|cochera|parking", case=False, na=False).astype(int)
df_ap["piso"]               = extraer_ap(r"[Pp]iso\s*([\d]+)")
df_ap["antiguedad"]         = extraer_ap(r"antiguedad[:\s]*([\d]+)")
df_ap["es_estreno"]         = texto_ap.str.contains(r"a estrenar|estreno|0 km", case=False, na=False).astype(int)

es_mono_ap = texto_ap.str.contains(r"monoambiente", case=False, na=False)
df_ap.loc[es_mono_ap & df_ap["rooms"].isna(), "rooms"] = 1
es_pb_ap = texto_ap.str.contains(r"planta baja", case=False, na=False)
df_ap.loc[es_pb_ap & df_ap["piso"].isna(), "piso"] = 0

# Imputaciones
mask_bath = df_ap["property_type"].str.lower().isin(["casa", "departamento"]) & (df_ap["bathrooms"].isna() | (df_ap["bathrooms"] == 0))
df_ap.loc[mask_bath, "bathrooms"] = 1
df_ap["bedrooms"]           = df_ap["bedrooms"].fillna(df_ap["rooms"] - 1)
df_ap["rooms"]              = df_ap["rooms"].fillna(df_ap["bedrooms"] + 1)
df_ap["bathrooms"]          = df_ap["bathrooms"].fillna(df_ap["rooms"] - df_ap["bedrooms"])
df_ap["surface_total_m2"]   = df_ap["surface_total_m2"].fillna(df_ap["surface_covered_m2"])
df_ap["surface_covered_m2"] = df_ap["surface_covered_m2"].fillna(df_ap["surface_total_m2"])
df_ap["pct_surface_covered"]= df_ap["surface_covered_m2"] / df_ap["surface_total_m2"]
df_ap["rooms"]              = df_ap["rooms"].fillna(1)
df_ap["bedrooms"]           = df_ap["bedrooms"].fillna(1)
df_ap["m2_por_ambiente"]    = df_ap["surface_total_m2"] / df_ap["rooms"].replace(0, float("nan"))

# Distancia al Obelisco
df_ap["dist_obelisco"] = np.sqrt(
    (df_ap["lat"] - OBELISCO_LAT) ** 2 + (df_ap["lon"] - OBELISCO_LON) ** 2
)

# Año y mes de publicación
_pub_ap = df_ap["publication_date"].copy().fillna("").astype(str)
df_ap["pub_year"]  = pd.to_numeric(_pub_ap.str.extract(r"(\d{4})")[0], errors="coerce").fillna(0).astype(int)
df_ap["pub_month"] = _pub_ap.str.lower().str.extract(r"(ene|feb|mar|abr|may|jun|jul|ago|sep|oct|nov|dic)")[0].map(months_map).fillna(0).astype(int)

# Target encoding usando gb y gb_l3 calculados sobre df_ent (sin leakage)
df_ap["price_m2_mean"] = list(zip(df_ap["property_type"], df_ap["location_2"]))
df_ap["price_m2_mean"] = df_ap["price_m2_mean"].map(gb).fillna(gb.mean())
df_ap["valor_estimado"] = df_ap["price_m2_mean"] * df_ap["surface_total_m2"]

df_ap["price_m2_mean_l3"] = list(zip(df_ap["property_type"], df_ap["location_3"]))
df_ap["price_m2_mean_l3"] = df_ap["price_m2_mean_l3"].map(gb_l3).fillna(gb_l3.mean())
df_ap["valor_estimado_l3"] = df_ap["price_m2_mean_l3"] * df_ap["surface_total_m2"]

In [ ]:
# Preddicíón final
X_ap = df_ap.select_dtypes("number").reindex(columns=X.columns, fill_value=0)
y_pred_ap = reg.predict(X_ap)

## 6. Generación del archivo para Kaggle

In [ ]:
# Guardar solución con índice original
nombre = "droideka"
pd.Series(y_pred_ap, index=ap_index, name="price").to_csv(f"../soluciones/solucion-{nombre}.csv", index_label="id")
print(f"Guardado: solucion-{nombre}.csv  ({len(y_pred_ap)} predicciones)")